In [22]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))

OpenAI key loaded: True


## RAG 1. Install dependencies

In [21]:
!python -m pip install llama-index llama-index-llms-openai llama-index-embeddings-openai openai

zsh:1: command not found: python


## RAG 2. Configuration

In [23]:
import os

# Path to the NEURON repo docs folder (relative to doc-project/)
DOCS_PATHS = [
    "../docs/nmodl",   
    "../docs/progref",
]

# Where to save the persistent index (so you don't re-index every time)
INDEX_STORE_PATH = "neuron_index"

## RAG 3. Build (or load) the index

In [24]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Configure models
Settings.llm = OpenAI(model="gpt-4o", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

if os.path.exists(INDEX_STORE_PATH):
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_STORE_PATH)
    index = load_index_from_storage(storage_context)
else:
    print("Building index from docs (this may take a few minutes)...")
    all_documents = []
    for docs_path in DOCS_PATHS:
        docs = SimpleDirectoryReader(
            input_dir=docs_path,
            recursive=True,
            required_exts=[".rst"],
        ).load_data()
        print(f"Loaded {len(docs)} .rst files from {docs_path}")
        all_documents.extend(docs)
    print(f"Total: {len(all_documents)} .rst files")
    index = VectorStoreIndex.from_documents(all_documents)
    index.storage_context.persist(persist_dir=INDEX_STORE_PATH)
    print("Index built and saved.")

Building index from docs (this may take a few minutes)...
Loaded 19 .rst files from ../docs/nmodl
Loaded 104 .rst files from ../docs/progref
Total: 123 .rst files


2026-03-18 10:28:08,539 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:09,964 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:11,538 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:15,122 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:16,786 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:17,710 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 10:28:18,521 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Index built and saved.


## RAG 4. Define the prompt template

In [25]:
PROMPT_TEMPLATE = """
You are a technical documentation assistant helping integrate community
Q&A content into the NEURON simulator's official documentation.

Below are the most relevant excerpts from the current NEURON documentation,
each labelled with its source .rst file path:

---------------------
{context_str}
---------------------

Here is a forum Q&A thread that contains information to be integrated:

<thread>
{query_str}
</thread>

Instructions:
- Identify every function, method, or class that the thread adds new information about.
- For each one, produce a unified diff showing what should be added to the relevant .rst file.
- Use standard unified diff format:
    --- a/<rst_file_path>
    +++ b/<rst_file_path>
    @@ -<line>,<count> +<line>,<count> @@
     (context lines with a leading space)
    +(new lines with a leading +)
- Base diffs on the existing documentation excerpts shown above.
- New content must match RST style (directives, inline code, section structure).
- Include code examples where relevant. Be concise and technical.
- If multiple files need changes, concatenate their diffs.

Return ONLY the unified diff text, with no preamble, explanation, or code fences.
Only include information clearly supported by the thread. Do not extrapolate.
"""

## RAG 5. Run a query for a single forum thread

In [28]:
import json
# Paste your forum thread here
with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    entry = json.load(f)

In [29]:
from llama_index.core import PromptTemplate

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=PromptTemplate(PROMPT_TEMPLATE),
)

def process_thread(thread_id, thread_text):
    """Query the index with a forum thread and return a unified diff string."""
    estimated_tokens = len(thread_text) // 4
    if estimated_tokens > 8000:
        print(f"WARNING: Thread {thread_id} is large (~{estimated_tokens} tokens) and may hit the token limit.")
    try:
        response = query_engine.query(thread_text)
        raw = str(response).strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return raw
    except Exception as e:
        error_msg = str(e)
        if "maximum context length" in error_msg or "token" in error_msg.lower():
            print(f"ERROR: Thread {thread_id} exceeded token limit (~{estimated_tokens} tokens).")
        else:
            print(f"ERROR: Thread {thread_id} failed: {error_msg}")
        return None


# Test on the first thread
forum_id = entry[1]["id"]
forum_thread = entry[1]["post"]

result = process_thread(forum_id, forum_thread)
print(result)

2026-03-18 11:01:46,572 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-18 11:01:50,861 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


--- a/Users/ryanzerbib/nrn/doc-project/../docs/progref/modelspec/programmatic/network/netcon.rst
+++ b/Users/ryanzerbib/nrn/doc-project/../docs/progref/modelspec/programmatic/network/netcon.rst
@@ -1,5 +1,18 @@
 Notice that this idiom allows recording from output cells 
             (which normally have no connecting netcons) as well as simplifying the 
             management of recording from cells. 
+        
+        
+            Note that NetCon.record() can also call a procedure, which can be used to 
+            track the number of spikes. For example:
+        
+            .. code-block:: none
+        
+                MAXNUM = 3
+                proc counter() {
+                  stv.append(t)  // record time of source event    
+                  if (stv.size()==MAXNUM) stoprun = 1
+                }
+        
+                nc.record("counter()")
         
         
             Note that NetCon.event(t) events are NOT recorded.


## RAG 6. Bulk processing

In [ ]:
import os

OUTPUT_DIR = "doc-project/diffs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for item in entry:
    thread_id = item['id']
    thread_text = item['post']
    print(f"Processing thread {thread_id}...")
    diff = process_thread(thread_id, thread_text)
    if diff:
        out_path = os.path.join(OUTPUT_DIR, f"thread_{thread_id}.diff")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(diff)
        print(f"  Saved: {out_path}")
    else:
        print(f"  Skipped thread {thread_id} (error or empty response).")

## RAG 7. Save results to a file for review

In [ ]:
# Diff files are saved to doc-project/diffs/thread_<id>.diff during bulk processing above.
# To apply a diff: patch -p1 < doc-project/diffs/thread_<id>.diff